In [344]:
import pandas as pd
import scipy.stats as stats
from scipy.stats import norm
import numpy as np

In [346]:
email_campaign_dataset=pd.read_excel("E:/Learning/Projects/Python_Projects/A by B Testing/realistic_ab_testing_email_campaign.xlsx")
email_campaign_dataset

,User_ID,Date,Variant,Emails_Sent,Email_Opened,Link_Clicked,Purchased,Revenue
0,11,2025-05-01,B,1,0,0,0,0.0
1,21,2025-05-01,A,1,0,0,0,0.0
2,22,2025-05-01,A,1,1,0,0,0.0
3,29,2025-05-01,A,1,0,0,0,0.0
4,34,2025-05-01,B,1,0,0,0,0.0
...,...,...,...,...,...,...,...,...
795,760,2025-05-15,B,1,1,0,0,0.0
796,783,2025-05-15,A,1,0,0,0,0.0
797,789,2025-05-15,B,1,0,0,0,0.0
798,791,2025-05-15,B,1,1,0,0,0.0


In [347]:
email_campaign_dataset.head(5)

,User_ID,Date,Variant,Emails_Sent,Email_Opened,Link_Clicked,Purchased,Revenue
0,11,2025-05-01,B,1,0,0,0,0.0
1,21,2025-05-01,A,1,0,0,0,0.0
2,22,2025-05-01,A,1,1,0,0,0.0
3,29,2025-05-01,A,1,0,0,0,0.0
4,34,2025-05-01,B,1,0,0,0,0.0


In [348]:
email_campaign_dataset.describe()

,User_ID,Emails_Sent,Email_Opened,Link_Clicked,Purchased,Revenue
count,800.0000,800.0,800.000000,800.00000,800.000000,800.000000
mean,400.5000,1.0,0.321250,0.04875,0.003750,0.666513
std,231.0844,0.0,0.467249,0.21548,0.061161,10.910303
min,1.0000,1.0,0.000000,0.00000,0.000000,0.000000
25%,200.7500,1.0,0.000000,0.00000,0.000000,0.000000
50%,400.5000,1.0,0.000000,0.00000,0.000000,0.000000
75%,600.2500,1.0,1.000000,0.00000,0.000000,0.000000
max,800.0000,1.0,1.000000,1.00000,1.000000,191.480000


In [349]:
Quick_Aggregations=(
    email_campaign_dataset.groupby("Variant").agg(
        total_emails_sent=("Emails_Sent","sum"),
        total_emails_opened=("Email_Opened","sum"),
        total_link_clciked=("Link_Clicked","sum"),
        total_purchases=("Purchased","sum"),
        total_revenue=("Revenue","sum")
        
    )
)
Quick_Aggregations

,total_emails_sent,total_emails_opened,total_link_clciked,total_purchases,total_revenue
Variant,,,,,
A,392,110,12,1,191.48
B,408,147,27,2,341.73


In [350]:
Conversion_Metrics= pd.DataFrame({
    "email_sent_to_email_opened": Quick_Aggregations.total_emails_opened*100 / Quick_Aggregations.total_emails_sent,
    "email_open_to_click_rate": Quick_Aggregations.total_link_clciked*100 / Quick_Aggregations.total_emails_opened,
    "email_click_to_purchase_rate": Quick_Aggregations.total_purchases*100 / Quick_Aggregations.total_link_clciked,
    "email_sent_to_purchase_rate": Quick_Aggregations.total_purchases / Quick_Aggregations.total_emails_sent
})

Conversion_Metrics

,email_sent_to_email_opened,email_open_to_click_rate,email_click_to_purchase_rate,email_sent_to_purchase_rate
Variant,,,,
A,28.061224,10.909091,8.333333,0.002551
B,36.029412,18.367347,7.407407,0.004902


### Hypothesis Testing(A/B Testing)

In [357]:
purchase_rate_a=Conversion_Metrics.loc['A','email_sent_to_purchase_rate']
purchase_rate_b=Conversion_Metrics.loc['B','email_sent_to_purchase_rate']
#h0= purchase_rate_a=purchase_rate_b #Null Hypothesis
#ha= purchase_rate_b>purchase_rate_a #Alternative Hypothesis
alpha=0.05
print(f"purchase rate of variant A= {purchase_rate_a}")
print(f"purchase rate of variant B= {purchase_rate_b}")

purchase rate of variant A= 0.002551020408163265
purchase rate of variant B= 0.004901960784313725


In [358]:
#pooled proportion
total_purchases_a=Quick_Aggregations.iloc[0,3]
total_purchases_b=Quick_Aggregations.iloc[1,3]
number_mails_sent_a=Quick_Aggregations.iloc[0,0]
number_mails_sent_b=Quick_Aggregations.iloc[1,0]
pooled_proportion=(total_purchases_a+total_purchases_b)/(number_mails_sent_a+number_mails_sent_b)
print(f"Pooled proportion of the test=",pooled_proportion)


Pooled proportion of the test= 0.00375


In [359]:
se=np.sqrt((pooled_proportion*(1-pooled_proportion)*(1/number_mails_sent_a + 1/number_mails_sent_b)))
print(f"StandardError=",se)

StandardError= 0.004322865064392592


In [360]:
z_test=(purchase_rate_a-purchase_rate_b)/se
print(f"Test Staitistic={z_test}")

Test Staitistic=-0.5438384823794614


In [361]:
critical_z=stats.norm.ppf(1-alpha)
print(f"Critical Value ={critical_z}")

Critical Value =1.6448536269514722


In [367]:
p_value=1-norm.cdf(z_test)
print(f"P Value for test={p_value}")

P Value for test=0.7067236875658922


In [369]:
if p_value<alpha:
    print("Reject null hypotheisi")
else:
    print("Fail to reject null hypothesis")

Fail to reject null hypothesis
